# Hand-Gestured Dust bin

## This project is basically a hand-gesture controlled dust bin, it combines both Image processing and Computer Vision to read the hand-gesture.

### Fist -> Come to User

### Palm -> Go back to initial position

### Others -> Do nothing

### real-time Image processing: Where we will try to make the image clearer by blurring the background and focusing on the hand

#### **1. We will start by Implementing our libraries that we will use for this project, and setting file paths which will be used often**

**Importing Libraries**

In [1]:
import os
import json
import pandas as pd
import cv2
import matplotlib.pyplot as plt
from tqdm import tqdm # For progress bar
from sklearn.model_selection import train_test_split
import shutil
from ultralytics import YOLO
import numpy as np

**Setting file paths**

In [3]:
Annotation_path = "./hagrid-sample-30k-384p/ann_train_val"
Images_path = "./hagrid-sample-30k-384p/hagrid_30k"

if os.path.exists(Annotation_path) and os.path.exists(Images_path):
    print("Paths exist.")

Paths exist.


# **--------------------------------------------------------------------------------------**

#### **2. Extract all the classes, and filter out the classes we are actually going to use which are fist and palm, and classify the rest as others**

**Extract all classes that are classified as "others" (NOT FIST OR PALM)**

In [4]:
all_classes = [] # To store all class names
if os.path.exists(Annotation_path): # Check if annotation path exists
    for filename in os.listdir(Annotation_path): # List all files in annotation directory
        if filename.endswith('.json'): # Process only JSON files
            all_classes.append(filename.split('.')[0]) # Get name without .json for example call.json -> call
else:
    print(f"ERROR: Annotation path not found: {Annotation_path}")
    # Handle error appropriately, maybe exit()

# Filter out 'fist' and 'palm'
other_classes = [cls for cls in all_classes if cls not in ['fist', 'palm']] # List comprehension to filter classes that aren't 'fist' or 'palm'

print(f"Found {len(other_classes)} other classes to process:") # Print number of other classes found
print(other_classes) # Print the list of other classes

Found 16 other classes to process:
['call', 'dislike', 'four', 'like', 'mute', 'ok', 'one', 'peace', 'peace_inverted', 'rock', 'stop', 'stop_inverted', 'three', 'three2', 'two_up', 'two_up_inverted']


**Checking if all files are there**

In [5]:
for class_name in other_classes: # Iterate over each class
    print(f"Processing class: {class_name}") # Print current class being processed
    class_json = os.path.join(Annotation_path, f"{class_name}.json") # Path to class JSON file
    class_images = os.path.join(Images_path, f"train_val_{class_name}") # Path to class images directory
        # Check if both paths exist
    if os.path.exists(class_json) and os.path.exists(class_images):
        print(f" Paths exist for class '{class_name}'.") # Confirm paths exist
    else:
        print(f" ERROR: Paths do not exist for class '{class_name}'.") # Error message if paths don't exist

Processing class: call
 Paths exist for class 'call'.
Processing class: dislike
 Paths exist for class 'dislike'.
Processing class: four
 Paths exist for class 'four'.
Processing class: like
 Paths exist for class 'like'.
Processing class: mute
 Paths exist for class 'mute'.
Processing class: ok
 Paths exist for class 'ok'.
Processing class: one
 Paths exist for class 'one'.
Processing class: peace
 Paths exist for class 'peace'.
Processing class: peace_inverted
 Paths exist for class 'peace_inverted'.
Processing class: rock
 Paths exist for class 'rock'.
Processing class: stop
 Paths exist for class 'stop'.
Processing class: stop_inverted
 Paths exist for class 'stop_inverted'.
Processing class: three
 Paths exist for class 'three'.
Processing class: three2
 Paths exist for class 'three2'.
Processing class: two_up
 Paths exist for class 'two_up'.
Processing class: two_up_inverted
 Paths exist for class 'two_up_inverted'.


**Make sure that fist and palm exist, along with their images**

In [6]:
fist_ann = os.path.join(Annotation_path, "fist.json") # Path to fist annotation file
palm_ann = os.path.join(Annotation_path, "palm.json") # Path to palm annotation file
if os.path.exists(fist_ann) and os.path.exists(palm_ann): # Check if both paths exist
    print("Paths exist.") 

fist_images = os.path.join(Images_path, "train_val_fist") # Path to fist images directory
palm_images = os.path.join(Images_path, "train_val_palm") # Path to palm images directory
if os.path.exists(fist_images) and os.path.exists(palm_images): # Check if both paths exist
    print("Paths exist.")

Paths exist.
Paths exist.


# **--------------------------------------------------------------------------------------**

#### **3. Understanding the json files we are actually dealing with**

**Loading all Json Files**

In [7]:
if os.path.exists(fist_ann):
    # File exists, now attempt to open and load
    with open(fist_ann, 'r') as f:
        # Note: This line can still crash if fist.json is invalid!
        fist_data = json.load(f) 
    print(f"Successfully opened and attempted to load {fist_ann}")
else:
    print(f"Error: The file '{fist_ann}' was not found.")

if os.path.exists(palm_ann):
    # File exists, now attempt to open and load
    with open(palm_ann, 'r') as f:
        # Note: This line can still crash if palm.json is invalid!
        palm_data = json.load(f) 
    print(f"Successfully opened and attempted to load {palm_ann}")
else:
    print(f"Error: The file '{palm_ann}' was not found.")

all_other_data = {} # Dictionary to store data for other classes
for class_name in other_classes:
    class_json = os.path.join(Annotation_path, f"{class_name}.json") # Path to class JSON file
    if os.path.exists(class_json):  # Check if file exists
        with open(class_json, 'r') as f: 
            class_data = json.load(f) # Load JSON data
            all_other_data[class_name] = class_data # Store in dictionary
        print(f"Successfully opened and attempted to load {class_json}") 
    else:
        print(f"Error: The file '{class_json}' was not found.")



Successfully opened and attempted to load ./hagrid-sample-30k-384p/ann_train_val\fist.json
Successfully opened and attempted to load ./hagrid-sample-30k-384p/ann_train_val\palm.json
Successfully opened and attempted to load ./hagrid-sample-30k-384p/ann_train_val\call.json
Successfully opened and attempted to load ./hagrid-sample-30k-384p/ann_train_val\dislike.json
Successfully opened and attempted to load ./hagrid-sample-30k-384p/ann_train_val\four.json
Successfully opened and attempted to load ./hagrid-sample-30k-384p/ann_train_val\like.json
Successfully opened and attempted to load ./hagrid-sample-30k-384p/ann_train_val\mute.json
Successfully opened and attempted to load ./hagrid-sample-30k-384p/ann_train_val\ok.json
Successfully opened and attempted to load ./hagrid-sample-30k-384p/ann_train_val\one.json
Successfully opened and attempted to load ./hagrid-sample-30k-384p/ann_train_val\peace.json
Successfully opened and attempted to load ./hagrid-sample-30k-384p/ann_train_val\peace_in

**Checking how Json file looks**

*This is where the annotation or the bounding box for the hand is*

In [8]:
count = 0
print("First 5 items:")
for key, value in fist_data.items():
    if count < 5:
        print(f"  Key: {key}")
        print(f"  Value: {value}\n") # Add a newline for readability
        count += 1
    else:
        break # Stop the loop after 5 items

First 5 items:
  Key: 003a238e-6ea2-48b0-ad4c-462f9dc655df
  Value: {'bboxes': [[0.62957251, 0.39665797, 0.20516206, 0.16012795]], 'labels': ['fist'], 'leading_hand': 'right', 'leading_conf': 0.8, 'user_id': 'b79fa61074d12fdac1e695233d33143715fa505c2bf6b90edd40a75107e89233'}

  Key: 006a26a3-12d2-411b-b980-0345365b560a
  Value: {'bboxes': [[0.57675642, 0.21875143, 0.17071956, 0.15190271]], 'labels': ['fist'], 'leading_hand': 'left', 'leading_conf': 1.0, 'user_id': 'ec7d929f1053e12d98c1c6af2fed292650453edb1dae513ab2e88fe48e512421'}

  Key: 00c9eb88-34c5-4350-b594-d30659ffb01c
  Value: {'bboxes': [[0.28766002, 0.38833272, 0.07709145, 0.06843053]], 'labels': ['fist'], 'leading_hand': 'right', 'leading_conf': 1.0, 'user_id': '25778395f8013ef97249f8a44cd12ef1c61e03d8cd2588c8b55281539bd29c00'}

  Key: 00f33929-6855-46b7-93db-4a3d86c91b50
  Value: {'bboxes': [[0.38599654, 0.87753338, 0.10116921, 0.04663836], [0.55235531, 0.50927987, 0.08750001, 0.05141174]], 'labels': ['no_gesture', 'fist'], 

In [9]:
count = 0
print("First 5 items:")
for key, value in palm_data.items():
    if count < 5:
        print(f"  Key: {key}")
        print(f"  Value: {value}\n") # Add a newline for readability
        count += 1
    else:
        break # Stop the loop after 5 items

First 5 items:
  Key: 00255531-cffe-4682-82da-47438a0e6fb4
  Value: {'bboxes': [[0.4818903, 0.72857142, 0.05622454, 0.0523392], [0.6319645, 0.50141545, 0.09697599, 0.07029632]], 'labels': ['no_gesture', 'palm'], 'leading_hand': 'left', 'leading_conf': 1.0, 'user_id': 'd39e22cd9343686f3f58b65aa45406de2942b876cb15f8308f2d44e4024bf613'}

  Key: 00453bde-c1ad-48ff-bad2-2141480dbcdc
  Value: {'bboxes': [[0.28236739, 0.17570206, 0.21128158, 0.18041136], [0.50747672, 0.61075935, 0.15925997, 0.09654546]], 'labels': ['palm', 'no_gesture'], 'leading_hand': 'right', 'leading_conf': 1.0, 'user_id': '597fe52313454664a0cab10aac85f3f31e46cf50e3b58fe745255c6465f06811'}

  Key: 0055c134-c63e-48af-b783-cbb7e50d5925
  Value: {'bboxes': [[0.3121809, 0.103207, 0.14617642, 0.12294465], [0.72193256, 0.51841276, 0.06800487, 0.0767783]], 'labels': ['palm', 'no_gesture'], 'leading_hand': 'right', 'leading_conf': 1.0, 'user_id': 'dc2cecbaa5bd7c07e79c8b6266b7f3d0229afd8c2576d915ac4e4180402695ff'}

  Key: 00642ef7

**for others, there is an array all_other_data, this array holds class name as Key, and value as in their full JSON data which is also a dictionary**

In [10]:
count = 0
print("First 5 items:")
for class_name, class_data in all_other_data.items():
    if count < 5:
        print(f"Class: {class_name}")
        sample_count = 0
        for key, value in class_data.items():
            if sample_count < 2:  # Print first 2 items of each class for brevity
                print(f"  Key: {key}")
                print(f"  Value: {value}\n") # Add a newline for readability
                sample_count += 1
            else:
                break
        count += 1
    else:
        break # Stop the loop after 5 classes

First 5 items:
Class: call
  Key: 01898f3e-8422-4e6a-a056-30206f905640
  Value: {'bboxes': [[0.42538839, 0.21308209, 0.05382926, 0.11273142]], 'labels': ['call'], 'leading_hand': 'right', 'leading_conf': 1.0, 'user_id': '01a1eea072d857da29fbe11a6dae84c8a48e320c24d98727352ae81b1ef6aaa4'}

  Key: 0516ab39-9dd3-41bf-9707-ccea0dbf985f
  Value: {'bboxes': [[0.40016984, 0.3223251, 0.13044141, 0.12870407]], 'labels': ['call'], 'leading_hand': 'right', 'leading_conf': 1.0, 'user_id': '894dcdb6beb82d2a93640a4078e4a8e3eef6bd15684a7a389aa1aeafb2bf48f3'}

Class: dislike
  Key: 001c6f56-85cf-4e45-bfc1-1af53c0e501b
  Value: {'bboxes': [[0.29848959, 0.56869285, 0.07580751, 0.06830282], [0.47884104, 0.3353942, 0.08987013, 0.09639664]], 'labels': ['no_gesture', 'dislike'], 'leading_hand': 'left', 'leading_conf': 1.0, 'user_id': '7c1d5778e9e3b4545386bed18a31ed20c7d32e18c90d296f3d78936126967297'}

  Key: 002b80a7-3292-4202-9a25-d6c8aac83a1e
  Value: {'bboxes': [[0.29465646, 0.84238796, 0.15554635, 0.0958

*We can see above that the Json file contains:* 

**Key:** Unique identifier of the image (image id)

**Values:**
1. bboxes: Normalized coordinates for the bounding box ([top-left-X-position, top-left-Y-position, width, height]).
2. labels: The object class name detected.
3. leading_hand: label identifying the active hand in the image.
4. leading_conf: the percentage confidence that the leading hand is the correct hand
5. user_id: user id of the person who contributed with the image

# **-------------------------------------------------------------------------------------**

### **4. Since we are using a sample dataset, there will be excess labels without any corresponding images; therefore, we will remove these entries to clean our data.**

*we are using a sample, the json is for the full dataset. Therefore we are removing the ones that aren't in the dataset we are using*

**Get fist data sample**

In [11]:
cleaned_fist_data = {} # Dictionary to hold cleaned fist data
for image_id, annotations in tqdm(fist_data.items(), desc="Cleaning Fist Data"): # Iterate over each image in fist data with progress bar
    image_path = os.path.join(fist_images, f"{image_id}.jpg") # Construct full image path
    # Check if the image file exists
    if os.path.exists(image_path):
        # If it exists, add it to the clean dictionary
        cleaned_fist_data[image_id] = annotations # Store valid entry

Cleaning Fist Data: 100%|██████████| 27764/27764 [00:00<00:00, 45764.27it/s]


**Get palm data sample**

In [12]:
cleaned_palm_data = {} # Dictionary to hold cleaned palm data
for image_id, annotations in tqdm(palm_data.items(), desc="Cleaning Palm Data"): # Iterate over each image in palm data with progress bar
    image_path = os.path.join(palm_images, f"{image_id}.jpg") # Construct full image path
    # Check if the image file exists
    if os.path.exists(image_path):
        # If it exists, add it to the clean dictionary
        cleaned_palm_data[image_id] = annotations # Store valid entry

Cleaning Palm Data: 100%|██████████| 28326/28326 [00:00<00:00, 44869.08it/s]


**Get others data sample**

In [13]:
cleaned_others_data = {} # Dictionary to hold cleaned other classes data 
for class_name, class_data in tqdm(all_other_data.items(), desc="Cleaning Other Classes Data"): # Iterate over each class with progress bar
    class_images = os.path.join(Images_path, f"train_val_{class_name}") # Path to class images directory
    cleaned_class_data = {} # Temporary dictionary for cleaned data of current class
    for image_id, annotations in tqdm(class_data.items(), desc = f"Cleaning {class_name} class data"): # Iterate over each image in the current class data
        image_path = os.path.join(class_images, f"{image_id}.jpg") # Construct full image path
        # Check if the image file exists
        if os.path.exists(image_path):
            # If it exists, add it to the clean dictionary
            cleaned_class_data[image_id] = annotations # Store valid entry
    cleaned_others_data[class_name] = cleaned_class_data # Store cleaned data for the current class     

Cleaning Other Classes Data: 100%|██████████| 16/16 [00:09<00:00,  1.66it/s]


**Check the results of our cleaning**

In [14]:
original_fist_count = len(fist_data)
clean_fist_count = len(cleaned_fist_data)
print(f"Original fist entries: {original_fist_count}")
print(f"Clean fist entries: {clean_fist_count} (Removed {original_fist_count - clean_fist_count})")
original_palm_count = len(palm_data)
clean_palm_count = len(cleaned_palm_data)
print(f"Original palm entries: {original_palm_count}")
print(f"Clean palm entries: {clean_palm_count} (Removed {original_palm_count - clean_palm_count})")
original_others_count = sum(len(class_data) for class_data in all_other_data.values()) # sum lengths of all other class data
clean_others_count = sum(len(class_data) for class_data in cleaned_others_data.values()) 
print(f"\nOriginal others entries checked: {original_others_count}") # Renamed variable for clarity
print(f"Clean others entries found: {clean_others_count} (Removed {original_others_count - clean_others_count})")

print ("TOTAL CLEANED ENTRIES:")
total_cleaned = clean_fist_count + clean_palm_count + clean_others_count
print (total_cleaned)

Original fist entries: 27764
Clean fist entries: 1735 (Removed 26029)
Original palm entries: 28326
Clean palm entries: 1770 (Removed 26556)

Original others entries checked: 453233
Clean others entries found: 28328 (Removed 424905)
TOTAL CLEANED ENTRIES:
31833


# **--------------------------------------------------------------------------------------**

### **5. Now that we are done with data prepping, it is time to transform it to Data frame for YOLO**

**Placing all clean data in a readable data frame for YOLO**

In [18]:
# getting all clean data in one place
all_data = []

# Process clean_fist_data
if cleaned_fist_data:
    for image_id, annotations in cleaned_fist_data.items():
        annotations['image_id'] = image_id
        annotations['class'] = 'fist'
        annotations['target_class'] = 'fist'
        all_data.append(annotations)
else:
    print("clean_fist_data is empty.")

# Process clean_palm_data
if cleaned_palm_data:
    for image_id, annotations in cleaned_palm_data.items():
        annotations['image_id'] = image_id
        annotations['class'] = 'palm'
        annotations['target_class'] = 'palm'
        all_data.append(annotations)
else:
    print("clean_palm_data is empty.")

if cleaned_others_data:
    for class_name, class_data in cleaned_others_data.items():
        for image_id, annotations in class_data.items():
            annotations['image_id'] = image_id
            annotations['class'] = class_name
            annotations['target_class'] = 'others'
            all_data.append(annotations)
else:
    print("clean_others_data is empty.")

# Create the final DataFrame
final_clean_df = pd.DataFrame(all_data)
print(f"Combined clean DataFrame created with {len(final_clean_df)} entries.")
final_clean_df.tail()

Combined clean DataFrame created with 31833 entries.


,bboxes,labels,leading_hand,leading_conf,user_id,image_id,class,target_class
31828,"[[0.5828369, 0.32664373, 0.14818502, 0.22770929]]",[two_up_inverted],left,1.0,348b9cc01d61ae211cdc0eb48b6087ee0fc1f9ff990142...,ff34a285-e9ae-4b7f-9a8e-6ea4fc126b8f,two_up_inverted,others
31829,"[[0.17860698, 0.29300424, 0.14429399, 0.224658...",[two_up_inverted],right,1.0,cff95f62e4b97189854944ca3a4f6ad60a0deef2dc56cc...,ff5bdf81-9b95-4e44-820d-a99ec37072b0,two_up_inverted,others
31830,"[[0.51612637, 0.21317261, 0.17052174, 0.294597...",[two_up_inverted],left,1.0,51cb30996bf5ccaa570dc37323273266a08337c797a75a...,ff8a6c64-54bf-4de2-a019-dae64c130340,two_up_inverted,others
31831,"[[0.26939816, 0.39967608, 0.15357095, 0.233924...",[two_up_inverted],right,1.0,a1b0bb3f79eb35269ae7e804eea8557fb1c7e97259cb8b...,ffb228d9-3470-4592-aade-c42543560ab7,two_up_inverted,others
31832,"[[0.68494436, 0.16356507, 0.06425648, 0.230024...",[two_up_inverted],left,1.0,5b0437fc8418f960a5efac042468e98b7c103c602b9de2...,ffd0d4cd-3364-41fe-b271-86137bdfe0f2,two_up_inverted,others


**Defining class maps and splitting the data frame**

*added stratify to maintain distribution as not all classes are of the same size*

In [16]:
# Define the mapping
class_map = {
    'fist': 0,
    'palm': 1,
    'others': 2
}

# Split the DataFrame (90% train, 10% validation)
# stratify ensures both train and val sets have a similar ratio of fist/palm
train_df, val_df = train_test_split(final_clean_df,
                                    test_size=0.1,
                                    random_state=42, # for reproducibility
                                    stratify=final_clean_df['class']) # Stratify by 'class' to maintain distribution

print(f"Training images: {len(train_df)}")
print(f"Validation images: {len(val_df)}")

Training images: 28649
Validation images: 3184


**Creating YOLO folder format**

In [25]:
# Define the root directory for your YOLO dataset (e.g., in your project folder)
# MAKE SURE 'yolo_dataset' folder DOES NOT already exist where you run this,
# or choose a different name.
yolo_dir = './yolo_dataset' # Use './' for current directory

# Define subdirectories
images_train_dir = os.path.join(yolo_dir, 'images', 'train')
images_val_dir = os.path.join(yolo_dir, 'images', 'val')
labels_train_dir = os.path.join(yolo_dir, 'labels', 'train')
labels_val_dir = os.path.join(yolo_dir, 'labels', 'val')

# Create the directories
os.makedirs(images_train_dir, exist_ok=True)
os.makedirs(images_val_dir, exist_ok=True)
os.makedirs(labels_train_dir, exist_ok=True)
os.makedirs(labels_val_dir, exist_ok=True)

print(f"Created YOLO folder structure inside: {yolo_dir}")

Created YOLO folder structure inside: ./yolo_dataset


**Create a format yolo data function, the main goal to take the cleaned and split annotation data (in DataFrame format) and transforms it into the specific file structure and content required for YOLO object detection training.**

In [26]:
# Function to process a DataFrame (train or val)
def format_yolo_data(df, split_name):
    print(f"\nProcessing {split_name} data...") # Print which split is being processed
    
    image_dir = os.path.join(yolo_dir, 'images', split_name) # Directory for images
    label_dir = os.path.join(yolo_dir, 'labels', split_name) # Directory for labels

    # Loop through each row in the DataFrame
    for _, row in tqdm(df.iterrows(), total=len(df), desc=f"Formatting {split_name}"): # Use tqdm for progress bar
        image_id = row['image_id'] # Image identifier
        class_name = row['target_class'] # Target class (fist, palm, others)
        org_class_name = row['class'] # Original class name
        bboxes = row['bboxes'] # List of bounding boxes for this image

        # Construct the SOURCE image path
        source_img_folder = os.path.join(Images_path, f"train_val_{org_class_name}") # Source folder based on original class
        source_img_path = os.path.join(source_img_folder, f"{image_id}.jpg") # Full source image path

        # Construct the DESTINATION paths 
        dest_img_path = os.path.join(image_dir, f"{image_id}.jpg") # Destination image path
        dest_label_path = os.path.join(label_dir, f"{image_id}.txt") # Destination label path

        #  Copy the image file 
        if os.path.exists(source_img_path): # Check if source image exists
            shutil.copyfile(source_img_path, dest_img_path) # Copy to destination
        else:
            print(f"Warning: Source image not found, skipping copy: {source_img_path}")
            continue # Skip processing labels if image is missing

        #  Create and write the label file 
        with open(dest_label_path, 'w') as f:
            for i, bbox in enumerate(bboxes):
                # Original format: [x_min, y_min, w, h] (normalized)
                x_min, y_min, w, h = bbox
                
                # Format for YOLO requires class_id and normalized bbox where [x_center, y_center, w, h]
                # We start by getting the label for THIS specific bbox
                
                label_for_bbox = row['labels'][i]
                #since we have other classes now, we want to map anything not in class_map to 'others'
                if label_for_bbox not in class_map: # if label not in our defined classes (fist or palm) just place it in others
                    # this will make sure all classes that arent fist or palm go to 'others'
                    label_for_bbox = 'others'

                # convert label to class_id (0: fist, 1: palm, 2: others) 
                class_id = class_map[label_for_bbox]
                
                # Convert to YOLO format [x_center, y_center, w, h]
                x_center = x_min + (w / 2)
                y_center = y_min + (h / 2)

                # Write the line to the label file
                f.write(f"{class_id} {x_center} {y_center} {w} {h}\n")


# --- Run the formatting for both sets ---
format_yolo_data(train_df, 'train')
format_yolo_data(val_df, 'val')

print("\nData formatting complete!")


Processing train data...


Formatting train: 100%|██████████| 28649/28649 [01:37<00:00, 292.88it/s]



Processing val data...


Formatting val: 100%|██████████| 3184/3184 [00:08<00:00, 372.59it/s]


Data formatting complete!


# **--------------------------------------------------------------------------------------**

### **6. Train the YOLO Model**

**TIME TO TRAIN MODEL**

In [ ]:
yaml_path = './yolo_dataset/dataset.yaml'
results_dir = './Hand_Gesture_Training'
model = YOLO('yolov8n.pt')  # Load a pre-trained YOLOv8n model

results = model.train(
    data=yaml_path,        
    epochs=30,           
    imgsz=384,           
    project=results_dir, 
    name='HG_final_run', 
    batch= 16,             
    workers= 0           # <<< NO PARALLEL DATA LOADING
)

print("\n--- Training Complete! ---")

New https://pypi.org/project/ultralytics/8.3.229 available  Update with 'pip install -U ultralytics'
Ultralytics 8.3.221  Python-3.12.7 torch-2.9.0+cu130 CUDA:0 (NVIDIA GeForce RTX 3050 Laptop GPU, 4096MiB)
engine\trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=./yolo_dataset/dataset.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=30, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=384, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=HG_final_run, n

**RESULTS FOR ALL IMAGES:**
1. P (Precision) = 0.989: Out of all the bounding boxes the model PREDICTED, $\mathbf{98.9\%}$ were correct.
2. R (Recall) = 0.978: Out of all the objects that ACTUALLY EXISTED, the model successfully found $\mathbf{97.8\%}$ of them.

To understand the other metrics, we will have to go through some stuff

**Intersection Over Union (IoU)**
The IoU is the first step. It determines how well the model's predicted bounding box overlaps with the Ground Truth (actual) bounding box.
Basically how close the predicted box IS TO THE ground truth bounding box
$$\text{IoU} = \frac{\text{Area of Overlap}}{\text{Area of Union}}$$
Rule: For a detection to even be considered a True Positive, its IoU must be greater than or equal to a set IoU Threshold (e.g., $0.50$).

**Average Precision (AP):** Average Precision is calculated per class (e.g., the AP for 'fist' or the AP for 'palm'). 

AP is the area under the Precision-Recall curve. 

This curve plots Precision against Recall at various confidence thresholds. It captures the trade-off: when the model is very confident (high threshold), Precision is high but Recall is low. When the model is less confident (low threshold), Recall is high but Precision drops.

The higher the AP, the better the model performs for that specific class.


**Mean Average Precision (mAP):** The "Mean" in mAP means you average the AP scores across all classes in your dataset.

$$\text{mAP} = \frac{1}{\text{N}_{\text{classes}}} \sum_{i=1}^{\text{N}_{\text{classes}}} \text{AP}_i$$

This provides a single number that summarizes the overall quality of the model across all categories.


3. Mean Average Precision (mAP50) = 0.992: model achieved $\mathbf{99.2\%}$ accuracy when requiring only a $\mathbf{50\%}$ overlap with the true box.
4. map50-95 = 0.872:


https://youtu.be/TSMJ-QRnk54?si=OKj1P_d7wKhO4M4q

# **--------------------------------------------------------------------------------------**

### **7. Preprocessing the frames before entering the model**

In [ ]:
def preprocess(frame, box):
    x1, y1, x2, y2 = box

    # Blur the entire frame
    blurred = cv2.GaussianBlur(frame, (41, 41), 0)

    # Copy the sharp hand region back from original frame
    blurred[y1:y2, x1:x2] = frame[y1:y2, x1:x2]

    return blurred

# **--------------------------------------------------------------------------------------**

## **The plan is to pipeline the model twice**

**First Pass**

We will let the model guess the hand, and its bounding box.

First pass pure use is to find where the hand is located so we could pre-process and make a more accurate prediction

**Pre-processing**

After finding the bounding box, we do preprocessing

**Second Pass**

We take the preprocessed hand, and guess which hand gesture it is

**First Pass -> Pre-processing -> Second Pass (where the prediction is made)**

### **8. Load the model and open the camera**

In [15]:
results_dir = './Hand_Gesture_Training'
model = YOLO(os.path.join(results_dir, 'HG_final_run', 'weights', 'best.pt'))

# Initialize video capture (0 for default camera)
cap = cv2.VideoCapture(0) # 0 is the default camera

while True:
    # 'ret' is a boolean: True if a frame was read successfully, False otherwise.
    # 'frame' is the actual frame (image) captured from the camera.
    ret, frame = cap.read()
    if not ret:
        print("Failed to grab frame")
        break
    
    firstpass1 = model(frame, verbose =False) # Perform inference on the frame

    fpannotated = firstpass1[0].boxes # Get the boxes from the first result
    
    if len(fpannotated) > 0: # If there are any boxes detected
        for box in fpannotated:
            # Extract box coordinates xmin, ymin, xmax, ymax
            x1, y1, x2, y2 = box.xyxy[0].cpu().numpy().astype(int) # Convert to integers
            
            preprocessed_roi = preprocess(frame, (x1, y1, x2, y2))


            # After preprocessing, perform second pass 

            secondpass = model(preprocessed_roi, verbose =False) # Second pass inference on the preprocessed ROI

            # Get the annotated result from the second pass
            if len(secondpass[0].boxes) > 0:
                
                # Get the first (highest confidence) detection
                gesture_box = secondpass[0].boxes[0]
                
                # Get the Class ID (number) and convert to Class Name (string)
                cls_id = int(gesture_box.cls[0]) # Class ID as integer
                class_name = model.names[cls_id] # Class name from model's names dictionary
                conf = float(gesture_box.conf[0]) # Confidence score as float

                # --- THE FILTER ---
                # Only draw and stitch if the class is NOT 'others'
                if class_name != 'others': 
                    
                    # 1. Create the annotated image (Draw the box)
                    spannotated = secondpass[0].plot()
                    
                    label_text = f"{class_name} {conf:.2f}"

                    (w, h), _ = cv2.getTextSize(label_text, cv2.FONT_HERSHEY_SIMPLEX, 0.6, 2)

                    # Draw a filled rectangle behind the text so it's readable
                    
                    if class_name == 'fist':
                        cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 0), 2) # Draw the bounding box
                        cv2.rectangle(frame, (x1, y1), (x1, y1), (0, 255, 0), -1)
                        
                    if class_name == 'palm':
                        cv2.rectangle(frame, (x1, y1), (x2, y2), (255, 0, 0), 2) # Draw the bounding box
                        cv2.rectangle(frame, (x1, y1), (x1, y1), (255, 0, 0), -1) # write the label

                    # Draw the text on the SHARP 'frame'
                    # We use (x1, y1 - 5) to put it just above the hand box
                    cv2.putText(frame, label_text, (x1, y1 - 5), 
                                cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 2)

            

            

        
    cv2.imshow("Hand Gesture Recognition", frame)
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()    

